# Paso 5 · Comparación de modelos de fraude

Se comparan Isolation Forest, XGBoost y LightGBM con la regresión logística balanceada del paso 4.
Todos usan los mismos cortes cronológicos y las mismas 29 variables del paso 3.
Se ordenan por precisión promedio (PR-AUC) de validación. La evaluación del ganador en prueba queda para el paso 6.

In [1]:
from importlib.metadata import version
from inspect import signature
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterSampler
from xgboost import XGBClassifier

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.data import load_raw, time_split, xy
from src.evaluate import summarize
from src.features import Preprocessor

SEED = 42
N_JOBS = 4

train, val, held_out = time_split(load_raw(project_root / "data/raw/creditcard.csv"))
assert train["Time"].max() < val["Time"].min()
assert val["Time"].max() < held_out["Time"].min()
del held_out
X_tr_raw, y_tr = xy(train)
X_va_raw, y_va = xy(val)
pre = Preprocessor().fit(X_tr_raw)
X_tr = pre.transform(X_tr_raw)
X_va = pre.transform(X_va_raw)
assert X_tr.columns.equals(X_va.columns) and X_tr.shape[1] == 29
assert y_tr.nunique() == y_va.nunique() == 2

train_prevalence = float(y_tr.mean())
pos_weight = float(y_tr.eq(0).sum() / y_tr.eq(1).sum())
print(f"Train: {len(y_tr):,} transacciones y {y_tr.sum()} fraudes")
print(f"Validación: {len(y_va):,} transacciones y {y_va.sum()} fraudes")
print(f"Prevalencia en train: {train_prevalence:.4%} · peso positivo: {pos_weight:.2f}")

Train: 198,607 transacciones y 366 fraudes
Validación: 42,558 transacciones y 55 fraudes
Prevalencia en train: 0.1843% · peso positivo: 541.64


## 5.1 · Isolation Forest y referencia logística

Isolation Forest ajusta sus árboles con las variables de train. La prevalencia de train
configura el corte interno de anomalía. La comparación usa el puntaje continuo
`-score_samples`, que aumenta con la rareza de la transacción.

In [2]:
logistic = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED)
logistic.fit(X_tr, y_tr)
isolation = IsolationForest(
    n_estimators=300, contamination=train_prevalence, random_state=SEED, n_jobs=N_JOBS
)
isolation.fit(X_tr)

models = {"logreg_balanced": logistic, "isolation_forest": isolation}
validation_scores = {
    "logreg_balanced": logistic.predict_proba(X_va)[:, 1],
    "isolation_forest": -isolation.score_samples(X_va),
}

## 5.2 · XGBoost

El peso de los positivos es la razón de operaciones legítimas a fraudes en train.
El entrenamiento usa un máximo de 2,000 árboles y una paciencia de 100 rondas
sobre `aucpr` de validación. La tabla final usa la misma precisión promedio
de scikit-learn para todos los modelos.

In [3]:
xgb_defaults = {
    "n_estimators": 2000,
    "learning_rate": 0.03,
    "max_depth": 4,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 2,
    "scale_pos_weight": pos_weight,
    "eval_metric": "aucpr",
    "early_stopping_rounds": 100,
    "tree_method": "hist",
    "random_state": SEED,
    "n_jobs": N_JOBS,
}

def fit_xgboost(features, labels, **overrides):
    model = XGBClassifier(**(xgb_defaults | overrides))
    return model.fit(features, labels, eval_set=[(X_va, y_va)], verbose=False)

xgb = fit_xgboost(X_tr, y_tr)
models["xgboost"] = xgb
validation_scores["xgboost"] = xgb.predict_proba(X_va)[:, 1]
print(f"Árboles en la mejor iteración: {xgb.best_iteration + 1}")

Árboles en la mejor iteración: 375


## 5.3 · LightGBM

LightGBM usa el mismo peso de positivos y detiene el entrenamiento
con 100 rondas de paciencia sobre la precisión promedio de validación.

In [4]:
lightgbm = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    metric="average_precision",
    deterministic=True,
    force_col_wise=True,
    random_state=SEED,
    n_jobs=N_JOBS,
    verbosity=-1,
)
# LightGBM 4.7 introduced eval_X/eval_y. Keep compatibility with 4.3–4.6.
if "eval_X" in signature(lightgbm.fit).parameters:
    validation_args = {"eval_X": X_va, "eval_y": y_va}
else:
    validation_args = {"eval_set": [(X_va, y_va)]}
lightgbm.fit(
    X_tr,
    y_tr,
    **validation_args,
    callbacks=[lgb.early_stopping(100, verbose=False)],
)
models["lightgbm"] = lightgbm
validation_scores["lightgbm"] = lightgbm.predict_proba(X_va)[:, 1]
print(f"Árboles en la mejor iteración: {lightgbm.best_iteration_}")

Árboles en la mejor iteración: 2


## 5.4 · Búsqueda aleatoria de XGBoost

Veinte combinaciones predefinidas por la semilla 42. El mejor intento se incorpora
a la comparación. La reutilización de validación para parada temprana y selección
puede elevar las métricas observadas. El paso 6 medirá la generalización en prueba.

In [5]:
search_space = {
    "max_depth": [3, 4, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05],
    "min_child_weight": [1, 2, 5],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
}
trials = []
best_ap = -np.inf
for trial, params in enumerate(
    ParameterSampler(search_space, n_iter=20, random_state=SEED), start=1
):
    candidate = fit_xgboost(X_tr, y_tr, **params)
    scores = candidate.predict_proba(X_va)[:, 1]
    ap = summarize(y_va, scores, "xgboost_tuned")["pr_auc"]
    trials.append({
        "trial": trial, **params, "trees": candidate.best_iteration + 1, "pr_auc": ap
    })
    if ap > best_ap:
        best_ap = ap
        models["xgboost_tuned"] = candidate
        validation_scores["xgboost_tuned"] = scores

tuning_results = pd.DataFrame(trials).sort_values("pr_auc", ascending=False, kind="stable")
tuning_results.round(6)

,trial,subsample,min_child_weight,max_depth,learning_rate,colsample_bytree,trees,pr_auc
18,19,0.8,5,6,0.01,0.6,919,0.864248
7,8,0.8,5,4,0.05,0.8,132,0.860940
11,12,1.0,1,6,0.03,0.8,708,0.859262
9,10,0.6,1,3,0.03,0.8,452,0.858897
0,1,0.6,5,6,0.01,0.8,982,0.858394
14,15,0.6,5,3,0.05,0.8,181,0.856975
13,14,0.8,5,4,0.01,0.8,446,0.855743
5,6,0.6,1,6,0.01,0.8,627,0.854140
17,18,1.0,5,3,0.01,0.8,564,0.853820
4,5,0.6,1,3,0.05,0.8,265,0.853351


## 5.6 · Manejo del desbalance

Se mantienen los hiperparámetros de XGBoost de 5.2 y se varía únicamente
la estrategia de desbalance. SMOTE genera ejemplos sintéticos dentro de train.
Las tres variantes se comparan sobre las mismas transacciones reales de validación.

In [6]:
unweighted = fit_xgboost(X_tr, y_tr, scale_pos_weight=1.0)
X_smote, y_smote = SMOTE(random_state=SEED).fit_resample(X_tr, y_tr)
smote_model = fit_xgboost(X_smote, y_smote, scale_pos_weight=1.0)
imbalance_results = pd.DataFrame([
    summarize(y_va, unweighted.predict_proba(X_va)[:, 1], "sin_correccion"),
    summarize(y_va, validation_scores["xgboost"], "peso_de_clase"),
    summarize(y_va, smote_model.predict_proba(X_va)[:, 1], "smote_solo_train"),
]).sort_values("pr_auc", ascending=False, kind="stable")
imbalance_results.round(6)

,model,pr_auc,roc_auc,recall@p80,recall@p50
0,sin_correccion,0.855215,0.985984,0.800000,0.854545
1,peso_de_clase,0.853195,0.987224,0.781818,0.854545
2,smote_solo_train,0.811796,0.974244,0.763636,0.800000


## 5.7 · Comparación y selección

La precisión promedio mide el orden de los puntajes de fraude. Como referencia,
un puntaje constante da la prevalencia de validación. Recall@p80 es el mejor
recall a una precisión de al menos 80% entre los cortes de validación.
La fila superior identifica el mejor candidato para el paso 6.

In [7]:
comparison = pd.DataFrame([
    summarize(y_va, scores, name) for name, scores in validation_scores.items()
]).sort_values("pr_auc", ascending=False, kind="stable", ignore_index=True)
winner = comparison.loc[0, "model"]
best = models[winner]
baseline_ap = comparison.loc[comparison["model"].eq("logreg_balanced"), "pr_auc"].item()
print(f"PR-AUC de referencia con puntaje constante: {y_va.mean():.6f}")
print(f"Ganador en validación: {winner} · mejora vs. logística: {comparison.loc[0, 'pr_auc'] - baseline_ap:+.6f}")
comparison.round(6)

PR-AUC de referencia con puntaje constante: 0.001292
Ganador en validación: xgboost_tuned · mejora vs. logística: +0.051424


,model,pr_auc,roc_auc,recall@p80,recall@p50
0,xgboost_tuned,0.864248,0.988384,0.818182,0.872727
1,xgboost,0.853195,0.987224,0.781818,0.854545
2,logreg_balanced,0.812824,0.980591,0.745455,0.854545
3,lightgbm,0.386226,0.907797,0.000000,0.636364
4,isolation_forest,0.033687,0.946454,0.000000,0.000000


In [8]:
model_dir = project_root / "models"
model_dir.mkdir(exist_ok=True)
model_path = model_dir / "fraud_model.joblib"
artifact = {
    "name": winner,
    "model": best,
    "preprocessor": pre,
    "feature_columns": X_tr.columns.to_list(),
    "validation_metrics": comparison.iloc[0].drop("model").to_dict(),
    "package_versions": {
        package: version(package)
        for package in ("pandas", "numpy", "scikit-learn", "xgboost", "lightgbm", "joblib")
    },
}
with TemporaryDirectory(dir=model_dir) as staging_dir:
    staged = Path(staging_dir) / model_path.name
    joblib.dump(artifact, staged, compress=3)
    loaded = joblib.load(staged)
    transformed = loaded["preprocessor"].transform(X_va_raw)
    assert transformed.columns.to_list() == loaded["feature_columns"]
    estimator = loaded["model"]
    if loaded["name"] == "isolation_forest":
        scores = -estimator.score_samples(transformed)
    else:
        scores = estimator.predict_proba(transformed)[:, 1]
    np.testing.assert_array_equal(scores, validation_scores[winner])
    staged.replace(model_path)

print(f"Modelo y preprocesador guardados en {model_path.relative_to(project_root)}.")
print("La recarga del archivo reproduce todos los puntajes de validación.")

Modelo y preprocesador guardados en models/fraud_model.joblib.
La recarga del archivo reproduce todos los puntajes de validación.
